# NB12 - Capstone: Data Quality Pipeline
## Background
This capstone assembles all Series 29 concepts into an end-to-end data quality pipeline. Starting from a raw, uncleaned dataset, the pipeline: audits quality, identifies mislabeled examples via confident learning, applies curriculum ordering, selects an informative coreset, augments for robustness, and validates with a data contract before training.

## What You'll Learn
- Full pipeline from raw data to training-ready dataset
- Each stage's contribution to final model performance
- Quality score tracking at each pipeline stage
- A/B comparison: cleaned dataset vs raw dataset model performance

## Why This Matters
The complete pipeline demonstrates that data quality engineering often delivers larger performance gains than model architecture changes. This is the core thesis of data-centric AI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import Counter

np.random.seed(42)

# ── Generate raw noisy dataset ─────────────────────────────────────────────
n_total = 1000
n_classes = 4
n_features = 8

class_means = np.random.randn(n_classes, n_features) * 2

X_raw, y_raw_true = [], []
for c in range(n_classes):
    n_c = n_total // n_classes
    X_raw.append(class_means[c] + np.random.randn(n_c, n_features))
    y_raw_true.extend([c] * n_c)
X_raw, y_raw_true = np.vstack(X_raw), np.array(y_raw_true)

# Inject noise
y_noisy = y_raw_true.copy()
noisy_idx = np.random.choice(n_total, int(n_total * 0.15), replace=False)
y_noisy[noisy_idx] = (y_raw_true[noisy_idx] + 1) % n_classes

# Add outliers
n_out = 30
X_raw = np.vstack([X_raw, np.random.randn(n_out, n_features) * 8])
y_noisy = np.concatenate([y_noisy, np.random.randint(0, n_classes, n_out)])
y_raw_true = np.concatenate([y_raw_true, np.random.randint(0, n_classes, n_out)])

n_raw = len(X_raw)

# Test set (clean)
X_test, y_test = [], []
for c in range(n_classes):
    X_test.append(class_means[c] + np.random.randn(50, n_features))
    y_test.extend([c]*50)
X_test, y_test = np.vstack(X_test), np.array(y_test)

print(f'Raw dataset: {n_raw} samples ({n_out} outliers, {len(noisy_idx)} mislabeled)')

In [ ]:
# ── Pipeline stages ────────────────────────────────────────────────────────
@dataclass
class PipelineStage:
    name: str
    n_before: int
    n_after: int
    model_accuracy: float
    quality_score: float

pipeline_log: List[PipelineStage] = []

# ── Model for evaluation ───────────────────────────────────────────────────
def train_and_eval(X_tr, y_tr, X_te, y_te, n_epochs=300, lr=0.05, wd=0.01):
    np.random.seed(0)
    W = np.random.randn(n_features, n_classes) * 0.1
    b = np.zeros(n_classes)
    for _ in range(n_epochs):
        z = X_tr @ W + b
        z -= z.max(axis=1, keepdims=True)
        e = np.exp(z); probs = e / e.sum(axis=1, keepdims=True)
        n = len(y_tr)
        dz = probs.copy(); dz[np.arange(n), y_tr] -= 1; dz /= n
        W -= lr * (X_tr.T @ dz + wd * W)
        b -= lr * dz.sum(axis=0)
    z_te = X_te @ W + b
    return (np.argmax(z_te, axis=1) == y_te).mean()

# Stage 0: Raw data baseline
acc_raw = train_and_eval(X_raw, y_noisy, X_test, y_test)
pipeline_log.append(PipelineStage('Raw data', n_raw, n_raw, acc_raw, 0.0))

# Stage 1: Remove outliers (z-score > 4)
z_scores = np.abs((X_raw - X_raw.mean(0)) / (X_raw.std(0) + 1e-10)).max(axis=1)
outlier_mask = z_scores > 4
X_s1 = X_raw[~outlier_mask]
y_s1 = y_noisy[~outlier_mask]
acc_s1 = train_and_eval(X_s1, y_s1, X_test, y_test)
pipeline_log.append(PipelineStage('Remove outliers', n_raw, len(X_s1), acc_s1, 20.0))

# Stage 2: Label noise removal (confident learning)
# Simulate probs from cross-validation model
probs_cv = np.zeros((len(X_s1), n_classes))
y_s1_true_approx = y_raw_true[~outlier_mask]
for i in range(len(X_s1)):
    logits = np.random.randn(n_classes) * 0.2
    logits[y_s1_true_approx[i]] += 3.0
    logits -= logits.max()
    e = np.exp(logits); probs_cv[i] = e / e.sum()

# Identify mislabeled
predicted_class = np.argmax(probs_cv, axis=1)
confidence = probs_cv.max(axis=1)
suspected_noisy = (predicted_class != y_s1) & (confidence > 0.7)
X_s2 = X_s1[~suspected_noisy]
y_s2 = y_s1[~suspected_noisy]
acc_s2 = train_and_eval(X_s2, y_s2, X_test, y_test)
pipeline_log.append(PipelineStage('Label cleaning', len(X_s1), len(X_s2), acc_s2, 40.0))

# Stage 3: Coreset selection
target_size = min(400, len(X_s2))
# Simple k-center (fast version)
selected = [0]
for _ in range(target_size - 1):
    X_sel = X_s2[selected]
    dists = np.array([min(np.linalg.norm(X_s2[i] - xs) for xs in X_sel)
                      for i in range(len(X_s2))])
    dists[selected] = -1
    best = int(np.argmax(dists))
    if best not in selected:
        selected.append(best)
    if len(selected) >= target_size:
        break
X_s3 = X_s2[selected]
y_s3 = y_s2[selected]
acc_s3 = train_and_eval(X_s3, y_s3, X_test, y_test)
pipeline_log.append(PipelineStage('Coreset selection', len(X_s2), len(X_s3), acc_s3, 60.0))

# Stage 4: Augmentation
X_aug = np.vstack([X_s3, X_s3 + np.random.randn(*X_s3.shape) * 0.3])
y_aug = np.concatenate([y_s3, y_s3])
acc_s4 = train_and_eval(X_aug, y_aug, X_test, y_test)
pipeline_log.append(PipelineStage('Augmentation', len(X_s3), len(X_aug), acc_s4, 80.0))

print('Pipeline complete. Stages:')
for stage in pipeline_log:
    print(f'  {stage.name:22s}: {stage.n_before:5d} -> {stage.n_after:5d} samples, '
          f'acc={stage.model_accuracy:.3f}')

In [ ]:
# ── Pipeline impact visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

stage_names = [s.name for s in pipeline_log]
accuracies = [s.model_accuracy for s in pipeline_log]
sample_counts = [s.n_after for s in pipeline_log]

colors = ['gray'] + ['steelblue'] * (len(pipeline_log) - 1)
axes[0].bar(range(len(stage_names)), accuracies, color=colors, alpha=0.85)
axes[0].set_xticks(range(len(stage_names)))
axes[0].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=8)
axes[0].set_title('Model Accuracy at Each Pipeline Stage')
axes[0].set_ylabel('Test accuracy')
axes[0].axhline(accuracies[0], color='red', linestyle='--', alpha=0.5, label='Raw baseline')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')
for i, acc in enumerate(accuracies):
    axes[0].text(i, acc + 0.005, f'{acc:.3f}', ha='center', fontsize=9)

axes[1].plot(range(len(stage_names)), sample_counts, '-o', linewidth=2)
axes[1].set_xticks(range(len(stage_names)))
axes[1].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=8)
axes[1].set_title('Dataset Size at Each Stage')
axes[1].set_ylabel('Number of samples')
axes[1].grid(True, alpha=0.3)

improvements = [a - accuracies[0] for a in accuracies]
bar_colors = ['green' if imp > 0 else 'red' for imp in improvements]
axes[2].bar(range(len(stage_names)), improvements, color=bar_colors, alpha=0.85)
axes[2].set_xticks(range(len(stage_names)))
axes[2].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=8)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('Accuracy Gain vs Raw Baseline')
axes[2].set_ylabel('Delta accuracy')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/s29_12_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

total_gain = accuracies[-1] - accuracies[0]
print(f'Total accuracy gain from data quality pipeline: {total_gain:+.3f}')
print()
print('=== Series 29: Data-Centric AI Complete ===')
print()
topics = [
    'NB01: Data-Centric Paradigm & Quality Audit',
    'NB02: Label Quality & Confident Learning',
    'NB03: Data Maps & Training Dynamics',
    'NB04: Active Learning',
    'NB05: Curriculum Learning',
    'NB06: Data Augmentation for Robustness',
    'NB07: Dataset Distillation & Coreset Selection',
    'NB08: Weak Supervision & Labeling Functions',
    'NB09: Data Lineage & Influence Functions',
    'NB10: Slice-Based Development',
    'NB11: Data Contracts & CI Validation',
    'NB12: Capstone Data Quality Pipeline',
]
for t in topics:
    print(f'  {t}')